# Fake Jobs – Exp 2: Enhanced Baseline-Modelle
- Gleiche Baselines auf verschiedenen Repräsentationen: cleaned vs. semantisch (PCA/no PCA) vs. enhanced (PCA/no PCA) vs. enhanced+semantisch
- Alignment über `row_id`, gemeinsamer 70/30-Split; feste Detektor-Params (fairer Vergleich)

In [5]:
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

## Repräsentationen laden (indexiert über row_id)
- Label aus cleaned (Outlier = `fraudulent == 1`); enhanced+semantisch = PCA30-Konkatenation

In [6]:
LABEL = "fraudulent"
ds = "fake_jobs"

def_load = lambda name: pd.read_csv(f"../../data/preprocessed/{name}_{ds}.csv").set_index("row_id")
cleaned = def_load("cleaned")
semantic_pca100 = def_load("semantic_pca100")
semantic_pca = def_load("semantic_pca30")
enhanced = def_load("enhanced")
enhanced_pca = def_load("enhanced_pca30")

reps = {
    "cleaned": cleaned.drop(columns=[LABEL]),
    "semantic_pca100": semantic_pca100.drop(columns=[LABEL]),
    "semantic_pca30": semantic_pca.drop(columns=[LABEL]),
    "enhanced": enhanced.drop(columns=[LABEL]),
    "enhanced_pca30": enhanced_pca.drop(columns=[LABEL]),
    "enhanced_semantic_pca30": enhanced_pca.drop(columns=[LABEL]).join(
        semantic_pca.drop(columns=[LABEL]), how="inner", lsuffix="_enh", rsuffix="_sem"),
}

## Gemeinsamer Index & Split
- Schnittmenge aller Repräsentationen (robust gegen unvollständige semantic-CSVs)

In [7]:
common = cleaned.index
for r in reps.values():
    common = common.intersection(r.index)
common = common.sort_values()
y = cleaned.loc[common, LABEL].values
print("common rows:", len(common), "outlier rate", round(y.mean(), 4))

tr_id, te_id = train_test_split(common, test_size=0.3, stratify=y, random_state=42)
y_train = cleaned.loc[tr_id, LABEL].values
y_test = cleaned.loc[te_id, LABEL].values

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("fake_jobs_experiment_2")

common rows: 17880 outlier rate 0.0484


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/fake_job_notebooks/exp2/../../mlruns/765377950431409952', creation_time=1780334077622, experiment_id='765377950431409952', last_update_time=1780334077622, lifecycle_stage='active', name='fake_jobs_experiment_2', tags={}, trace_location=None, workspace='default'>

## Detektoren x Repräsentationen
- Beste Hyperparameter aus Exp 1 (README), **kein GridSearch**; AutoEncoder One-Class (nur Inlier, GPU)

In [8]:
# beste Hyperparameter aus Experiment 1 (README) — kein GridSearch in Exp 2
detectors = {
    "iforest": (IForest, {"n_estimators": 100, "max_features": 1.0, "random_state": 42}, False),
    "loda": (LODA, {"n_bins": 20, "n_random_cuts": 200}, False),
    "ecod": (ECOD, {}, False),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [64, 32], "epoch_num": 20, "random_state": 42, "device": "cuda"}, True),
}

for rep_name, rep in reps.items():
    Xtr = rep.loc[tr_id].values
    Xte = rep.loc[te_id].values
    for det_name, (Model, params, inlier_only) in detectors.items():
        t0 = time.perf_counter()
        Xfit = Xtr[y_train == 0] if inlier_only else Xtr
        model = Model(**params)
        model.fit(Xfit)
        scores = model.decision_function(Xte)
        runtime = time.perf_counter() - t0
        ap = average_precision_score(y_test, scores)
        auc = roc_auc_score(y_test, scores)
        with mlflow.start_run(run_name=f"{rep_name}__{det_name}"):
            mlflow.log_param("representation", rep_name)
            mlflow.log_param("detector", det_name)
            mlflow.log_param("n_features", rep.shape[1])
            mlflow.log_metric("average_precision", ap)
            mlflow.log_metric("auc_roc", auc)
            mlflow.log_metric("runtime_s", runtime)
        print(f"{rep_name:24s} {det_name:12s} AP={ap:.4f} AUC={auc:.4f} feat={rep.shape[1]} t={runtime:.1f}s")

cleaned                  iforest      AP=0.0876 AUC=0.6141 feat=13 t=0.3s
cleaned                  loda         AP=0.0432 AUC=0.4705 feat=13 t=0.2s
cleaned                  ecod         AP=0.0759 AUC=0.6125 feat=13 t=0.1s


Training: 100%|██████████| 20/20 [00:20<00:00,  1.02s/it]


cleaned                  autoencoder  AP=0.0803 AUC=0.6298 feat=13 t=21.1s
semantic_pca100          iforest      AP=0.0570 AUC=0.5378 feat=139 t=0.3s
semantic_pca100          loda         AP=0.0452 AUC=0.4832 feat=139 t=0.4s
semantic_pca100          ecod         AP=0.0460 AUC=0.4903 feat=139 t=2.5s


Training: 100%|██████████| 20/20 [00:20<00:00,  1.05s/it]


semantic_pca100          autoencoder  AP=0.0818 AUC=0.5732 feat=139 t=21.6s
semantic_pca30           iforest      AP=0.0869 AUC=0.6197 feat=69 t=0.3s
semantic_pca30           loda         AP=0.0387 AUC=0.4303 feat=69 t=0.3s
semantic_pca30           ecod         AP=0.0587 AUC=0.5503 feat=69 t=0.3s


...

enhanced                 autoencoder  AP=0.6078 AUC=0.8806 feat=512 t=22.0s
enhanced_pca30           iforest      AP=0.1496 AUC=0.7071 feat=30 t=0.3s
enhanced_pca30           loda         AP=0.1237 AUC=0.7154 feat=30 t=0.2s
enhanced_pca30           ecod         AP=0.1100 AUC=0.6975 feat=30 t=0.1s


Training: 100%|██████████| 20/20 [00:19<00:00,  1.01it/s]


enhanced_pca30           autoencoder  AP=0.6272 AUC=0.8923 feat=30 t=20.4s
enhanced_semantic_pca30  iforest      AP=0.1109 AUC=0.6793 feat=99 t=0.3s
enhanced_semantic_pca30  loda         AP=0.0448 AUC=0.5056 feat=99 t=0.8s
enhanced_semantic_pca30  ecod         AP=0.0825 AUC=0.6579 feat=99 t=0.4s


Training: 100%|██████████| 20/20 [00:19<00:00,  1.00it/s]


enhanced_semantic_pca30  autoencoder  AP=0.1983 AUC=0.8176 feat=99 t=21.0s
